# 01 — CMMS Data Exploration

Exploratory analysis of the synthetic CMMS dataset that feeds the turnaround
optimizer: the **asset master**, **failure history**, and **work order**
tables produced by `src.utils.data_generator`.

This notebook answers three questions before any modeling happens:
1. What does the asset population look like (classes, ages, replacement value)?
2. What does the work-order backlog look like (cost, craft-hours, priority mix)?
3. Are there any data-quality red flags worth fixing upstream?


In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

from src.utils.config import DATA_RAW
from src.etl.extract import load_work_orders, load_asset_master, load_failure_history

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 160)

assets   = load_asset_master()
failures = load_failure_history()
wos      = load_work_orders()

print(f"Assets:   {len(assets):,} rows")
print(f"Failures: {len(failures):,} rows")
print(f"Work Orders: {len(wos):,} rows")

2026-06-24 01:37:01  INFO      [etl.extract]  Extracting asset master from /home/claude/turnaround-optimizer/data/raw/asset_master.csv


2026-06-24 01:37:01  INFO      [etl.extract]    → 623 assets


2026-06-24 01:37:01  INFO      [helpers]  ⏱  load_asset_master completed in 0.011 s


2026-06-24 01:37:01  INFO      [etl.extract]  Extracting failure history from /home/claude/turnaround-optimizer/data/raw/failure_history.csv


2026-06-24 01:37:01  INFO      [etl.extract]    → 2467 failure events


2026-06-24 01:37:01  INFO      [helpers]  ⏱  load_failure_history completed in 0.010 s


2026-06-24 01:37:01  INFO      [etl.extract]  Extracting work orders from /home/claude/turnaround-optimizer/data/raw/work_orders.csv


2026-06-24 01:37:01  INFO      [etl.extract]    → 550 rows | 25 columns


2026-06-24 01:37:01  INFO      [helpers]  ⏱  load_work_orders completed in 0.008 s


Assets:   623 rows
Failures: 2,467 rows
Work Orders: 550 rows


## 1. Asset Master Overview

In [2]:
assets.head()

,asset_tag,asset_class,asset_name,area,install_date,age_days,weibull_beta,weibull_eta,c_safety,c_env,c_prod,c_cost,replace_usd
0,PMP-0001,PMP,Centrifugal Pump,Offsites,2014-12-04,4319,2.5,1825,3,2,4,3,48000
1,PMP-0002,PMP,Centrifugal Pump,Utilities,2013-09-27,4752,2.5,1825,3,2,4,3,48000
2,PMP-0003,PMP,Centrifugal Pump,Unit-200,2015-12-30,3928,2.5,1825,3,2,4,3,48000
3,PMP-0004,PMP,Centrifugal Pump,Unit-100,2012-02-07,5350,2.5,1825,3,2,4,3,48000
4,PMP-0005,PMP,Centrifugal Pump,Tank-Farm,2015-02-07,4254,2.5,1825,3,2,4,3,48000


In [3]:
class_summary = (
    assets.groupby("asset_class")
    .agg(count=("asset_tag", "count"),
         mean_age_yrs=("age_days", lambda s: (s / 365).mean()),
         total_replace_value=("replace_usd", "sum"))
    .sort_values("total_replace_value", ascending=False)
    .round(1)
)
class_summary

,count,mean_age_yrs,total_replace_value
asset_class,,,
TWR,12,6.7,11760000
CMP,18,7.9,8820000
VSL,28,10.1,8120000
TNK,20,8.6,7800000
HX,35,7.7,6825000
PPL,45,8.1,3960000
PMP,80,8.0,3840000
ELEC,85,7.3,2040000
VLV,120,8.0,1620000


In [4]:
fig = px.bar(
    class_summary.reset_index(),
    x="asset_class", y="total_replace_value",
    title="Total Replacement Value at Risk by Equipment Class",
    labels={"total_replace_value": "Replacement Value (USD)", "asset_class": "Equipment Class"},
    text_auto=".2s",
)
fig.update_layout(template="plotly_white", height=420)
fig.show()

In [5]:
fig = px.histogram(
    assets, x="age_days", color="asset_class", nbins=40,
    title="Asset Age Distribution (days in service)",
    labels={"age_days": "Age (days)"},
)
fig.update_layout(template="plotly_white", height=420, barmode="overlay")
fig.update_traces(opacity=0.65)
fig.show()

## 2. Failure History Overview

In [6]:
print(f"Mean time-to-failure across all records: {failures.time_to_failure_d.mean():.0f} days")
print(f"Median: {failures.time_to_failure_d.median():.0f} days")
failures.failure_mode.value_counts()

Mean time-to-failure across all records: 1499 days
Median: 1091 days


failure_mode
Corrosion          334
Fouling            327
Blockage           320
Leakage            307
Wear               302
Seal Failure       297
Fatigue            292
Bearing Failure    288
Name: count, dtype: int64

In [7]:
fig = px.box(
    failures, x="asset_class", y="time_to_failure_d", color="asset_class",
    title="Time-to-Failure Distribution by Equipment Class",
    labels={"time_to_failure_d": "Time to Failure (days)"},
)
fig.update_layout(template="plotly_white", height=450, showlegend=False)
fig.show()

## 3. Work Order Backlog Overview

In [8]:
print(f"Total estimated cost of ALL {len(wos)} work orders: "
      f"${wos.estimated_cost_usd.sum():,.0f}")
print(f"Total craft hours requested: {wos.total_craft_hours.sum():,.0f} h")
print(f"Mandatory tasks: {wos.mandatory.sum()} "
      f"({wos.mandatory.mean()*100:.1f}% of backlog)")

wos.groupby("priority").agg(
    count=("wo_id", "count"),
    total_cost=("estimated_cost_usd", "sum"),
).sort_values("total_cost", ascending=False)

Total estimated cost of ALL 550 work orders: $30,527,016
Total craft hours requested: 31,136 h
Mandatory tasks: 49 (8.9% of backlog)


,count,total_cost
priority,,
Medium,187,10761946.31
High,157,7797078.89
Critical,117,6722544.96
Low,89,5245445.68


In [9]:
fig = go.Figure()
fig.add_trace(go.Histogram(x=wos.estimated_cost_usd, nbinsx=50, marker_color="#3b82f6"))
fig.update_layout(
    title="Work Order Cost Distribution (full backlog, before optimization)",
    xaxis_title="Estimated Cost (USD)", yaxis_title="Count",
    template="plotly_white", height=420,
)
fig.show()

print(f"\nNote: total backlog cost (${wos.estimated_cost_usd.sum():,.0f}) vastly exceeds any "
      f"realistic turnaround budget — this is exactly WHY a selection optimizer is needed "
      f"rather than 'just do everything'.")


Note: total backlog cost ($30,527,016) vastly exceeds any realistic turnaround budget — this is exactly WHY a selection optimizer is needed rather than 'just do everything'.


In [10]:
task_type_cost = wos.groupby("task_type").estimated_cost_usd.sum().sort_values(ascending=False)
fig = px.pie(
    values=task_type_cost.values, names=task_type_cost.index,
    title="Backlog Cost Share by Task Type", hole=0.45,
)
fig.update_layout(template="plotly_white", height=420)
fig.show()

## 4. Data Quality Checks

Quick sanity checks before this data is trusted by the optimizer.


In [11]:
issues = []

if (wos.estimated_cost_usd < 0).any():
    issues.append(f"{(wos.estimated_cost_usd < 0).sum()} work orders with negative cost")
if wos.mech_hours.isna().any():
    issues.append(f"{wos.mech_hours.isna().sum()} work orders with missing mech_hours")
orphan_preds = wos[wos.predecessor_wo_id.notna() & ~wos.predecessor_wo_id.isin(wos.wo_id)]
if len(orphan_preds):
    issues.append(f"{len(orphan_preds)} work orders reference an unknown predecessor")
orphan_assets = wos[~wos.asset_tag.isin(assets.asset_tag)]
if len(orphan_assets):
    issues.append(f"{len(orphan_assets)} work orders reference an unknown asset_tag")

if issues:
    print("⚠️  Data quality issues found (the ETL transform stage will handle these):")
    for i in issues:
        print(f"   - {i}")
else:
    print("✅ No data quality issues detected in this sample.")

✅ No data quality issues detected in this sample.


## Summary

The synthetic backlog totals roughly 10–15× the typical turnaround budget in
estimated cost — consistent with real-world plant turnarounds, where the CMMS
backlog always vastly exceeds available budget and craft-hour capacity. This
confirms a *selection* problem, not a *scheduling* problem alone: the 0-1
knapsack ILP in `02_weibull_reliability_analysis.ipynb` →
`03_optimization_walkthrough.ipynb` is the right tool for deciding **which**
subset of work orders to execute now versus defer to the next cycle.
